# Import

In [1]:
import glob
import json
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

# Stramlit用にデータを加工

In [2]:
def create_demo_datasetA(datasetA_path: str) -> pl.DataFrame:
    """HUNOB2024データセットAを読み込み、緯度・経度、詳細時刻、uidごとの方位情報を追加する"""

    # CSV読み込み
    df_a = pl.read_csv(datasetA_path)
    df_a = df_a.filter(pl.col("uid")==0)

    # 緯度経度情報の追加（例: 緯度 = x*0.005 + 34.497, 経度 = y*0.005 + 136.5）
    add_lat_col = (pl.col("x")*0.005 + 34.497).alias("latitude")
    add_lon_col = (pl.col("y")*0.005 + 136.5).alias("longitude")
    df_a = df_a.with_columns(add_lat_col).with_columns(add_lon_col)

    # 詳細時刻の追加
    # d: 0～74, 0 が 2020/01/05 を表す
    # t: 0～47, 0 が 0時、以降30分間隔
    start_date = pl.lit("2020-01-05T00:00:00").str.strptime(pl.Datetime, format="%Y-%m-%dT%H:%M:%S")
    df_a = df_a.with_columns(
        (
            start_date
            + pl.col("d") * pl.duration(days=1)
            + pl.col("t") * pl.duration(minutes=30)
        ).cast(pl.Datetime).alias("datetime")
    )

    # datetime 列を "YYYY-MM-DD HH:mm" の文字列に変換
    # df_a = df_a.with_columns(
    #     pl.col("datetime").dt.strftime("%Y-%m-%d %H:%M").alias("datetime_str")
    # )

    # uid ごとに、日時順にソート
    df_a = df_a.sort(["uid", "datetime"])

    # 同一 uid 内で、前の行の緯度・経度を取得（最初の行はnullになる）
    df_a = df_a.with_columns(
        pl.col("latitude").shift(1).over("uid").alias("prev_lat"),
        pl.col("longitude").shift(1).over("uid").alias("prev_lon")
    )
    
    # 前の地点と現在の地点から方位を計算する関数
    def calculate_bearing(row: dict) -> float:
        prev_lat = row["prev_lat"]
        prev_lon = row["prev_lon"]
        lat = row["latitude"]
        lon = row["longitude"]
        # 最初の行は前の位置がないので None を返す
        if prev_lat is None or prev_lon is None:
            return None
        # 度をラジアンに変換
        lat1 = np.radians(prev_lat)
        lon1 = np.radians(prev_lon)
        lat2 = np.radians(lat)
        lon2 = np.radians(lon)
        delta_lon = lon2 - lon1
        x = np.sin(delta_lon) * np.cos(lat2)
        y = np.cos(lat1) * np.sin(lat2) - (np.sin(lat1) * np.cos(lat2) * np.cos(delta_lon))
        initial_bearing = np.arctan2(x, y)
        initial_bearing = np.degrees(initial_bearing)
        # 0～360度に正規化
        bearing = (initial_bearing + 360) % 360
        return bearing

    # 各行ごとに、前の緯度・経度と現在の緯度・経度から方位を計算
    df_a = df_a.with_columns(
        pl.struct(["prev_lat", "prev_lon", "latitude", "longitude"]).apply(calculate_bearing).alias("bearing")
    )

    return df_a

demo_df = create_demo_datasetA("/kaggle/s3storage/01_public/humob-challenge-2024/input/cityA_groundtruthdata.csv.gz")

display(demo_df.head(10))
print(len(demo_df))

/tmp/ipykernel_38971/171919016.py:64: DeprecationWarning: `apply` is deprecated. It has been renamed to `map_elements`.
  pl.struct(["prev_lat", "prev_lon", "latitude", "longitude"]).apply(calculate_bearing).alias("bearing")


uid,d,t,x,y,latitude,longitude,datetime,prev_lat,prev_lon,bearing
i64,i64,i64,i64,i64,f64,f64,datetime[μs],f64,f64,f64
0,0,1,79,86,34.892,136.93,2020-01-05 00:30:00,null,null,null
0,0,2,79,86,34.892,136.93,2020-01-05 01:00:00,34.892,136.93,0.0
0,0,8,77,86,34.882,136.93,2020-01-05 04:00:00,34.892,136.93,180.0
0,0,9,77,86,34.882,136.93,2020-01-05 04:30:00,34.882,136.93,0.0
0,0,19,81,89,34.902,136.945,2020-01-05 09:30:00,34.882,136.93,31.594451
0,0,20,82,88,34.907,136.94,2020-01-05 10:00:00,34.902,136.945,320.646014
0,0,21,81,89,34.902,136.945,2020-01-05 10:30:00,34.907,136.94,140.643153
0,0,22,81,89,34.902,136.945,2020-01-05 11:00:00,34.902,136.945,0.0
0,0,24,76,86,34.877,136.93,2020-01-05 12:00:00,34.902,136.945,206.208581


1261


In [3]:
demo_df.write_csv("demo_dataset.csv")